# Genetic Algorithm — Hands-On Laboratory

**Based on the supplied laboratory manual** · Python built-ins only

This notebook is divided into short cells so each concept can be run and inspected independently. It covers animal-chromosome decoding, fitness, mutation, crossover, roulette-wheel selection, a complete Genetic Algorithm (GA), and feature selection with the UCI Statlog (Heart) dataset.

**Before Part G:** download `heart.dat` from the [UCI Statlog (Heart) dataset page](https://archive.ics.uci.edu/dataset/145/statlog%2Bheart) and place it in the same folder as this notebook. Earlier sections run without that file; Part G will skip its experiment if the file is missing.

> Educational exercise only. The heart-disease classifier is not a medical diagnostic system and must not be used for clinical decisions.

## 1. Setup

In [ ]:
import random

random.seed(42)

## Part A — Decode animal characteristics

Each gene represents one characteristic. A value of `1` means the characteristic is present; `0` means it is absent.

In [ ]:
characteristics = [
    "Tall",
    "Heavy",
    "Fast",
    "Has horns",
    "Lives in water"
]

chromosome = [1, 0, 1, 1, 0]
print(chromosome)

In [ ]:
def decode_animal(chromosome):
    description = []

    for i in range(len(chromosome)):
        if chromosome[i] == 1:
            description.append(characteristics[i])
        else:
            description.append("Not " + characteristics[i])

    return description

In [ ]:
print(decode_animal(chromosome))

In [ ]:
exercise_chromosomes = ["10110", "01101", "11001", "00100"]

for bits in exercise_chromosomes:
    animal = [int(bit) for bit in bits]
    print(bits, "->", decode_animal(animal))

### Exercise A3 — Create your own animal
Edit these labels and bits to define a different animal chromosome.

In [ ]:
my_characteristics = ["Can fly", "Has fur", "Lays eggs", "Lives in water", "Has stripes"]
my_chromosome = [1, 0, 1, 0, 1]

for label, bit in zip(my_characteristics, my_chromosome):
    print(label + ":", "Yes" if bit == 1 else "No")

### Exercise A5 — Prepare for mutation
The third gene is at index `2` in Python. Changing it from `1` to `0` changes the **Fast** characteristic.

In [ ]:
before_mutation = [1, 0, 1, 1, 0]
after_mutation = before_mutation[:]
after_mutation[2] = 1 - after_mutation[2]

print("Before:", before_mutation, "->", decode_animal(before_mutation))
print("After :", after_mutation, "->", decode_animal(after_mutation))

## Part B — Fitness functions

For animal chromosomes, fitness is the number of genes that match a target chromosome.

In [ ]:
def animal_fitness(chromosome, target):
    score = 0

    for i in range(len(chromosome)):
        if chromosome[i] == target[i]:
            score += 1

    return score

In [ ]:
target_animal = [1, 0, 1, 1, 0]
example_animal = [1, 0, 0, 1, 0]

print("Fitness:", animal_fitness(example_animal, target_animal), "out of", len(target_animal))

### Numerical chromosome decoding

The complete GA uses a six-bit chromosome to represent an integer. This decoder builds the value one bit at a time; it does not use `int(bits, 2)`.

In [ ]:
def decode(chromosome):
    value = 0

    for bit in chromosome:
        value = value * 2 + bit

    return value

In [ ]:
print(decode([1, 1, 0, 0, 1, 0]))  # 50

### Exercise B2 — Objective function

The objective rewards values close to `x = 50`.

In [ ]:
def objective(x):
    return 100 - abs(x - 50)


sample_population = [
    [1, 1, 0, 0, 1, 0],
    [0, 1, 1, 0, 0, 1],
    [1, 1, 1, 1, 1, 1]
]

for candidate in sample_population:
    x = decode(candidate)
    print(candidate, "x =", x, "fitness =", objective(x))

## Part C — Mutation

Bit-flip mutation changes a selected `0` to `1` or a selected `1` to `0`.

In [ ]:
def mutate_one(chromosome, position):
    child = chromosome[:]
    child[position] = 1 - child[position]
    return child

In [ ]:
test_chromosome = [1, 0, 1, 1, 0, 1]

print("Before:", test_chromosome)
print("Position 2:", mutate_one(test_chromosome, 2))
print("Position 0:", mutate_one(test_chromosome, 0))
print("Position 5:", mutate_one(test_chromosome, 5))

In [ ]:
def mutate(chromosome, mutation_rate):
    child = chromosome[:]

    for i in range(len(child)):
        if random.random() < mutation_rate:
            child[i] = 1 - child[i]

    return child

In [ ]:
random.seed(42)

for _ in range(5):
    print(mutate(test_chromosome, 0.20))

### Exercise C3 — Compare mutation rates

The number of flipped bits at intermediate probabilities varies randomly. A rate of `0.00` flips none and `1.00` flips every bit.

In [ ]:
for rate in [0.00, 0.10, 0.50, 1.00]:
    random.seed(42)
    child = mutate(test_chromosome, rate)
    changed_bits = sum(1 for a, b in zip(test_chromosome, child) if a != b)
    print("Mutation rate:", rate, "| Changed bits:", changed_bits, "| Result:", child)

## Part D — Single-point crossover

Crossover takes the front of each parent up to a cut point and swaps the remaining tails.

In [ ]:
def single_point_crossover(parent1, parent2, point=None):
    if len(parent1) != len(parent2):
        raise ValueError("Parents must have the same chromosome length")
    if len(parent1) < 2:
        raise ValueError("Chromosomes must contain at least two genes")

    if point is None:
        point = random.randint(1, len(parent1) - 1)

    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2, point

In [ ]:
parent1 = [1, 1, 0, 1, 0, 1]
parent2 = [0, 0, 1, 0, 1, 1]

child1, child2, cut_point = single_point_crossover(parent1, parent2)
print("Parent 1:", parent1)
print("Parent 2:", parent2)
print("Cut point:", cut_point)
print("Child 1 :", child1)
print("Child 2 :", child2)

### Exercise D1 — Manual crossover examples

In [ ]:
crossover_examples = [
    ([1, 1, 0, 1, 0, 1], [0, 0, 1, 1, 1, 0], 3),
    ([1, 1, 1, 0, 0, 0], [0, 1, 0, 1, 1, 1], 4),
    ([1, 0, 0, 1, 1, 1], [0, 1, 1, 0, 0, 1], 2)
]

for p1, p2, point in crossover_examples:
    c1, c2, _ = single_point_crossover(p1, p2, point)
    print("Parents:", p1, p2, "cut =", point, "-> children:", c1, c2)

### Exercise D3 — Crossover probability

In [ ]:
def crossover_with_probability(parent1, parent2, crossover_rate):
    if random.random() >= crossover_rate:
        return parent1[:], parent2[:]

    child1, child2, _ = single_point_crossover(parent1, parent2)
    return child1, child2

In [ ]:
random.seed(42)
print("Rate 0.0:", crossover_with_probability(parent1, parent2, 0.0))
print("Rate 1.0:", crossover_with_probability(parent1, parent2, 1.0))

## Part E — Roulette-wheel selection

A chromosome with a larger fitness has a higher chance of selection. The probabilities below use the manual’s example fitness values.

In [ ]:
selection_population = ["A", "B", "C", "D"]
selection_fitnesses = [10, 20, 30, 40]
total_fitness = sum(selection_fitnesses)
cumulative = 0

for chromosome, fit in zip(selection_population, selection_fitnesses):
    probability = fit / total_fitness
    cumulative += probability
    print(chromosome, "fitness =", fit,
          "probability =", round(probability, 2),
          "cumulative =", round(cumulative, 2))

In [ ]:
def roulette_selection(population, fitnesses):
    if len(population) != len(fitnesses) or not population:
        raise ValueError("Population and fitnesses must have the same non-zero length")

    total = sum(fitnesses)
    if total <= 0:
        return random.choice(population)[:]

    r = random.random() * total
    cumulative = 0

    for chromosome, fit in zip(population, fitnesses):
        cumulative += fit
        if cumulative >= r:
            return chromosome[:] if isinstance(chromosome, list) else chromosome

    last = population[-1]
    return last[:] if isinstance(last, list) else last

In [ ]:
binary_population = [[0, 0, 0, 1], [0, 0, 1, 0], [0, 1, 0, 0], [1, 0, 0, 0]]
binary_fitnesses = [10, 20, 30, 40]

random.seed(42)
for _ in range(10):
    print(roulette_selection(binary_population, binary_fitnesses))

In [ ]:
def select_population(population, fitnesses, size):
    new_population = []

    for _ in range(size):
        new_population.append(roulette_selection(population, fitnesses))

    return new_population


new_population = select_population(binary_population, binary_fitnesses, 8)
print(new_population)

## Part F — Build a complete Genetic Algorithm

The GA searches for an integer close to 50 using six-bit chromosomes, roulette selection, single-point crossover, and bit-flip mutation.

In [ ]:
CHROMOSOME_LENGTH = 6
POPULATION_SIZE = 10
GENERATIONS = 20
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.80

In [ ]:
def create_chromosome():
    chromosome = []
    for _ in range(CHROMOSOME_LENGTH):
        chromosome.append(random.randint(0, 1))
    return chromosome


def create_population(size=POPULATION_SIZE):
    population = []
    for _ in range(size):
        population.append(create_chromosome())
    return population

In [ ]:
def fitness(chromosome):
    x = decode(chromosome)
    return objective(x)

In [ ]:
def run_basic_ga(population_size=10, generations=20,
                 mutation_rate=0.05, crossover_rate=0.80,
                 seed=42, show_history=True):
    random.seed(seed)
    population = create_population(population_size)
    best_overall = None
    best_overall_fitness = float("-inf")
    history = []

    for generation in range(generations):
        fitnesses = [fitness(candidate) for candidate in population]
        best_index = fitnesses.index(max(fitnesses))
        best = population[best_index][:]
        best_fit = fitnesses[best_index]

        if best_fit > best_overall_fitness:
            best_overall = best[:]
            best_overall_fitness = best_fit

        history.append(best_fit)
        if show_history:
            print("Generation:", generation,
                  "Best x:", decode(best),
                  "Fitness:", best_fit)

        new_population = []
        while len(new_population) < population_size:
            parent1 = roulette_selection(population, fitnesses)
            parent2 = roulette_selection(population, fitnesses)
            child1, child2 = crossover_with_probability(
                parent1, parent2, crossover_rate
            )
            child1 = mutate(child1, mutation_rate)
            child2 = mutate(child2, mutation_rate)
            new_population.append(child1)

            if len(new_population) < population_size:
                new_population.append(child2)

        population = new_population

    final_fitnesses = [fitness(candidate) for candidate in population]
    final_index = final_fitnesses.index(max(final_fitnesses))
    final_candidate = population[final_index]

    if final_fitnesses[final_index] > best_overall_fitness:
        best_overall = final_candidate[:]
        best_overall_fitness = final_fitnesses[final_index]

    print("Final best chromosome:", best_overall)
    print("Final decoded value:", decode(best_overall))
    print("Final fitness:", best_overall_fitness)
    return best_overall, history

In [ ]:
best_chromosome, fitness_history = run_basic_ga()

### Exercise F2 — Experiment with parameters

Each experiment uses the same random seed for a more consistent comparison. Because a GA is stochastic, a single run does not guarantee the global optimum.

In [ ]:
experiments = [
    (10, 20, 0.05, 0.80),
    (20, 20, 0.05, 0.80),
    (10, 50, 0.05, 0.80),
    (10, 50, 0.20, 0.80),
    (10, 50, 0.05, 1.00)
]

for pop_size, generations, mutation_rate, crossover_rate in experiments:
    best, _ = run_basic_ga(
        population_size=pop_size,
        generations=generations,
        mutation_rate=mutation_rate,
        crossover_rate=crossover_rate,
        seed=42,
        show_history=False
    )
    print("Population:", pop_size,
          "Generations:", generations,
          "Mutation:", mutation_rate,
          "Crossover:", crossover_rate,
          "Best x:", decode(best))

## Part G — Real-world application: Heart-disease feature selection

Each chromosome contains 13 bits, one per input feature. A `1` selects the feature and `0` leaves it out. The fitness combines 3-nearest-neighbor test accuracy with a small penalty for using more features.

### G1. Load `heart.dat`

The manual specifies the UCI Statlog (Heart) text file, with 13 input values followed by target `1` (absence) or `2` (presence). Keep the file beside this notebook.

In [ ]:
def load_heart_data(filename):
    X = []
    y = []

    with open(filename, "r") as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) < 14:
                continue

            values = [float(value) for value in parts[:13]]
            target = int(float(parts[13]))
            target = 0 if target == 1 else 1

            X.append(values)
            y.append(target)

    return X, y

In [ ]:
HEART_DATA_AVAILABLE = False
X_raw = []
y = []

try:
    X_raw, y = load_heart_data("heart.dat")
    if not X_raw:
        print("No valid records found in heart.dat. Check the file format.")
    else:
        HEART_DATA_AVAILABLE = True
        print("Number of records:", len(X_raw))
        print("Number of features:", len(X_raw[0]))
        print("First record:", X_raw[0])
        print("First target:", y[0])
except FileNotFoundError:
    print("heart.dat was not found. Download it from the UCI link in the introduction,")
    print("place it beside this notebook, then rerun this cell and the Part G cells.")

### G2. Normalize numeric columns

This uses min-max normalization as shown in the manual. The manual normalizes the full dataset before splitting it into training and testing sets.

In [ ]:
def normalize_columns(X):
    if not X:
        return []

    rows = len(X)
    cols = len(X[0])
    minimum = [X[0][j] for j in range(cols)]
    maximum = [X[0][j] for j in range(cols)]

    for i in range(rows):
        for j in range(cols):
            if X[i][j] < minimum[j]:
                minimum[j] = X[i][j]
            if X[i][j] > maximum[j]:
                maximum[j] = X[i][j]

    normalized = []
    for i in range(rows):
        row = []
        for j in range(cols):
            if maximum[j] == minimum[j]:
                row.append(0.0)
            else:
                value = (X[i][j] - minimum[j]) / (maximum[j] - minimum[j])
                row.append(value)
        normalized.append(row)

    return normalized

In [ ]:
if HEART_DATA_AVAILABLE:
    X = normalize_columns(X_raw)

    indices = list(range(len(X)))
    random.seed(42)
    random.shuffle(indices)
    split = int(0.80 * len(indices))

    train_indices = indices[:split]
    test_indices = indices[split:]
    X_train = [X[i] for i in train_indices]
    y_train = [y[i] for i in train_indices]
    X_test = [X[i] for i in test_indices]
    y_test = [y[i] for i in test_indices]

    print("Training records:", len(X_train))
    print("Testing records :", len(X_test))
else:
    print("Skipped: load heart.dat to prepare the train/test split.")

### G3. Implement 3-nearest-neighbor classification

Distances are calculated using only features whose chromosome bit is `1`.

In [ ]:
def distance(a, b, selected):
    total = 0.0

    for j in range(len(a)):
        if selected[j] == 1:
            difference = a[j] - b[j]
            total += difference * difference

    return total ** 0.5

In [ ]:
def predict_one(test_row, X_train, y_train, selected, k=3):
    distances = []

    for i in range(len(X_train)):
        d = distance(test_row, X_train[i], selected)
        distances.append((d, y_train[i]))

    distances.sort()
    votes = {}

    for i in range(min(k, len(distances))):
        label = distances[i][1]
        votes[label] = votes.get(label, 0) + 1

    best_label = None
    best_votes = -1
    for label, count in votes.items():
        if count > best_votes:
            best_votes = count
            best_label = label

    return best_label

In [ ]:
def accuracy(X_test, y_test, X_train, y_train, selected):
    if not X_test:
        return 0.0

    correct = 0
    for i in range(len(X_test)):
        prediction = predict_one(
            X_test[i], X_train, y_train, selected, k=3
        )
        if prediction == y_test[i]:
            correct += 1

    return correct / len(y_test)

### G4. Feature-selection fitness

Fitness is `accuracy - penalty`, where the penalty is `0.05 × (selected features / total features)`. A chromosome with no selected features receives fitness `0`.

In [ ]:
FEATURES = 13
FEATURE_PENALTY = 0.05
POPULATION_SIZE_FEATURES = 12
GENERATIONS_FEATURES = 10
MUTATION_RATE_FEATURES = 0.05
CROSSOVER_RATE_FEATURES = 0.80


def feature_fitness(chromosome):
    selected_count = sum(chromosome)
    if selected_count == 0:
        return 0.0

    acc = accuracy(X_test, y_test, X_train, y_train, chromosome)
    penalty = FEATURE_PENALTY * (selected_count / len(chromosome))
    return acc - penalty

### G5. GA operators for feature selection

In [ ]:
def create_feature_chromosome():
    chromosome = []
    for _ in range(FEATURES):
        chromosome.append(random.randint(0, 1))

    if sum(chromosome) == 0:
        chromosome[random.randint(0, FEATURES - 1)] = 1

    return chromosome


def create_feature_population():
    return [create_feature_chromosome() for _ in range(POPULATION_SIZE_FEATURES)]

In [ ]:
def roulette_feature_selection(population, fitnesses):
    minimum = min(fitnesses)
    adjusted = [value - minimum + 0.000001 for value in fitnesses]
    total = sum(adjusted)
    r = random.random() * total
    cumulative = 0.0

    for chromosome, fit in zip(population, adjusted):
        cumulative += fit
        if cumulative >= r:
            return chromosome[:]

    return population[-1][:]

In [ ]:
def crossover_features(parent1, parent2):
    if random.random() > CROSSOVER_RATE_FEATURES:
        return parent1[:], parent2[:]

    point = random.randint(1, FEATURES - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2


def mutate_features(chromosome):
    child = chromosome[:]

    for i in range(len(child)):
        if random.random() < MUTATION_RATE_FEATURES:
            child[i] = 1 - child[i]

    if sum(child) == 0:
        child[random.randint(0, FEATURES - 1)] = 1

    return child

### G6. Run feature selection

The settings match the lab manual. The search can take longer than the earlier examples because every candidate is evaluated with a simple Python k-NN classifier.

In [ ]:
best_overall = None
best_overall_fitness = -1.0

if HEART_DATA_AVAILABLE:
    random.seed(42)
    population = create_feature_population()

    for generation in range(GENERATIONS_FEATURES):
        fitnesses = [feature_fitness(candidate) for candidate in population]
        best_index = fitnesses.index(max(fitnesses))
        best = population[best_index]
        best_fit = fitnesses[best_index]

        if best_fit > best_overall_fitness:
            best_overall = best[:]
            best_overall_fitness = best_fit

        print("Generation:", generation,
              "Fitness:", round(best_fit, 4),
              "Features:", sum(best),
              "Chromosome:", best)

        new_population = []
        while len(new_population) < POPULATION_SIZE_FEATURES:
            parent1 = roulette_feature_selection(population, fitnesses)
            parent2 = roulette_feature_selection(population, fitnesses)
            child1, child2 = crossover_features(parent1, parent2)
            child1 = mutate_features(child1)
            child2 = mutate_features(child2)
            new_population.append(child1)

            if len(new_population) < POPULATION_SIZE_FEATURES:
                new_population.append(child2)

        population = new_population

    print("\nBEST FEATURE SUBSET")
    print("Chromosome:", best_overall)
    print("Number of features:", sum(best_overall))
    print("Fitness:", round(best_overall_fitness, 4))
else:
    print("Skipped: load heart.dat to run feature selection.")

### G7. Map chromosome positions to feature names and compare accuracy

In [ ]:
feature_names = [
    "age", "sex", "chest_pain", "resting_bp", "cholesterol",
    "fasting_blood_sugar", "resting_ecg", "max_heart_rate",
    "exercise_angina", "oldpeak", "slope", "major_vessels", "thal"
]

if HEART_DATA_AVAILABLE and best_overall is not None:
    print("Selected features:")
    for i in range(len(best_overall)):
        if best_overall[i] == 1:
            print("-", feature_names[i])

    all_features = [1] * FEATURES
    all_accuracy = accuracy(X_test, y_test, X_train, y_train, all_features)
    ga_accuracy = accuracy(X_test, y_test, X_train, y_train, best_overall)

    print("Accuracy using all features:", round(all_accuracy, 4))
    print("Accuracy using GA features :", round(ga_accuracy, 4))
    print("Features using all        :", sum(all_features))
    print("Features using GA         :", sum(best_overall))
else:
    print("Skipped: the dataset and a GA-selected chromosome are required.")

## Student analysis questions

After running Part G with `heart.dat`, record your answers:

1. Did the GA select all 13 features or a smaller subset?
2. What was the best chromosome found in your run?
3. What was the classification accuracy using all features?
4. What was the accuracy using the GA-selected subset?
5. Did reducing the number of features reduce, maintain, or improve accuracy?
6. How did changing the mutation rate affect the result?
7. How did changing population size affect the result?
8. How did changing the number of generations affect the result?
9. Run the experiment three times with different random seeds. Did you obtain the same subset? Why might results differ?
10. Why is it useful for fitness to consider both predictive performance and the number of selected features?

## Laboratory submission checklist

- [ ] Animal decoder implemented without binary-string-to-integer conversion.
- [ ] Fitness function implemented.
- [ ] Bit-flip mutation implemented.
- [ ] Single-point crossover implemented.
- [ ] Roulette-wheel selection implemented.
- [ ] Complete GA implemented and parameters experimented with.
- [ ] UCI Heart dataset loaded without Pandas.
- [ ] Feature-selection chromosome implemented.
- [ ] GA-selected feature subset reported.
- [ ] Accuracy with all features compared against GA-selected features.

## Key takeaway

A Genetic Algorithm can be built from lists, loops, functions, and random numbers. The same operators used to optimize a small numerical objective can also search for useful subsets of features in a classification task.